# E04: 
we saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W?

# Solution:

In [1]:
import torch
import torch.nn.functional as F

In [2]:
words = open("names.txt", 'r').read().splitlines()
words[:5], len(words)

(['emma', 'olivia', 'ava', 'isabella', 'sophia'], 32033)

In [3]:
min(words, key=len), len(min(words, key=len)), max(words, key=len), len(max(words, key=len))

('an', 2, 'muhammadibrahim', 15)

In [4]:
# Get list of all the characters
chars = ['.'] + sorted(list(set(''.join(words)))) # ., a, b, c, d, .... x, y, z

# Create dictionary for mapping single character to its ID
stoi = {s:i for i, s in enumerate(chars)}

# Create another dictionary for reverse mapping
itos = {i:s for s, i in stoi.items()}

# Check if its correct
len(chars), stoi['.'], stoi['a'], itos[0]

(27, 0, 1, '.')

In [5]:
# Get all 27*27 combinations (26 alphabets and one '.')
all_combinations = [a + b for a in chars for b in chars]

# Create dictionary for mapping combination to its ID
ctoi={c:i for i,c in enumerate(all_combinations)}

# Create another dictionary for reverse mapping
itoc={i:c for c, i in ctoi.items()}

# Check if its correct
len(ctoi), ctoi['..'], itoc[1]

(729, 0, '.a')

## Create train, dev and test split

In [6]:
g1 = torch.Generator().manual_seed(42)
perm = torch.randperm(len(words), generator=g1)
perm, perm.shape

(tensor([ 4348, 12372,  7029,  ...,  7956,  2399,  8375]), torch.Size([32033]))

In [7]:
words_shuffled = [words[i] for i in perm]
train_words = words_shuffled[:25626]
dev_words   = words_shuffled[25626:28830]
test_words  = words_shuffled[28830:32033]

## Trigram

In [8]:
# Creating the dataset
def create_dataset_trigram(words):
    xs, ys = [], []
    for w in words:
        chs = ['.','.'] + list(w) + ['.']  # eg ['.', '.', 'e', 'm', 'm', 'a', '.']
        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            ch12 = ch1+ch2
            idx12 = ctoi[ch12]
            idx3 = stoi[ch3]
            xs.append(idx12)
            ys.append(idx3)
    
    # Converting dataset into tensor
    xs = torch.tensor(xs)
    ys = torch.tensor(ys)
    num = xs.numel()
    return xs, ys, num 

In [9]:
xs_train_t, ys_train_t, training_size_t = create_dataset_trigram(train_words)
xs_dev_t, ys_dev_t, dev_size_t = create_dataset_trigram(dev_words)
xs_test_t, ys_test_t, test_size_t = create_dataset_trigram(test_words)

In [10]:
print(xs_dev_t.shape)
print(ys_dev_t.shape)
print(dev_size_t)

torch.Size([22768])
torch.Size([22768])
22768


In [15]:
# CUDA 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# Move data and weights to device
xs_train_t = xs_train_t.to(device)
ys_train_t = ys_train_t.to(device)
xs_dev_t = xs_dev_t.to(device)
ys_dev_t = ys_dev_t.to(device)
xs_test_t = xs_test_t.to(device)
ys_test_t = ys_test_t.to(device)


Using: cuda


In [22]:
g = torch.Generator(device=device).manual_seed(312312)
W_trigram = torch.randn((729, 27), generator=g, device=device, requires_grad=True)
W_trigram = W_trigram.to(device)

## Directly Indexing instead of one_hot

In [25]:
lambdaa = 0.5

for i in range(100):

    # Forward pass
    logits = W_trigram[xs_train_t] #(729, 27)[N] → (N, 27)

    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)

    loss = (-probs[torch.arange(training_size_t, device=device),ys_train_t].log().mean()
        + lambdaa / (2 * training_size_t) * (W_trigram ** 2).sum()
    )

    print(f"Train loss: {loss.item()}")

    # Backward
    W_trigram.grad = None
    loss.backward()

    # Update
    with torch.no_grad():
        W_trigram += -70 * W_trigram.grad

Train loss: 3.705507278442383
Train loss: 3.584233283996582
Train loss: 3.5022659301757812
Train loss: 3.432533025741577
Train loss: 3.3701553344726562
Train loss: 3.3137264251708984
Train loss: 3.26257061958313
Train loss: 3.2161991596221924
Train loss: 3.1741693019866943
Train loss: 3.136047601699829
Train loss: 3.1013991832733154
Train loss: 3.0697970390319824
Train loss: 3.0408458709716797
Train loss: 3.014195680618286
Train loss: 2.9895520210266113
Train loss: 2.9666683673858643
Train loss: 2.9453423023223877
Train loss: 2.9254050254821777
Train loss: 2.9067139625549316
Train loss: 2.8891475200653076
Train loss: 2.8726000785827637
Train loss: 2.8569798469543457
Train loss: 2.8422045707702637
Train loss: 2.8282032012939453
Train loss: 2.8149099349975586
Train loss: 2.80226731300354
Train loss: 2.790222406387329
Train loss: 2.77872896194458
Train loss: 2.767744541168213
Train loss: 2.7572309970855713
Train loss: 2.74715518951416
Train loss: 2.73748517036438
Train loss: 2.72819447517

In [29]:
# Generating names
for i in range(30):
    out = [] # for storing the output
    context = ['.', '.'] # for triggering the model to start outputting characters to form a name
    
    while True:
        pair = context[0] + context[1]
        ix = ctoi[pair]
    
        logits = W_trigram[ix].unsqueeze(0)
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)

        # Randomly draw one index from the 27 indices, using the values in p as the sampling probabilities.
        # idx with highest probability has higher chances, but its not guaranteed
        next_ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    
        next_char = itos[next_ix]

        if next_char == '.': 
            break         # breaking when it hits end character
        out.append(next_char)
        context = [context[1], next_char]
    
    if len(out)>1:
        print(''.join(out)) 

tauous
launi
andaleifdwbkno
sequetwci
amir
maylemdulcjmnsonwqwmlelloe
los
malynelumbtmgfvpe
cabelynna
lhwth
la
maloycprgghlmvvydcob
kaodgjrpaejkwavanyianna
chanxo
kail
backsgvncmjszjbarrynee
rea
breslettwnjvvvtuz
afcore
ariah
alain
zayanna
fvzkjdvgu
minna
fzmpkbbaston
ellenijlndrinspnwwnopoluna
taliupkbzizen
wkqsdine
ee


## Evaluating Trigram

In [30]:
# Evaluating on test_set 
logits = W_trigram[xs_test_t]
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
logprobs = torch.log(probs)
nlls = -logprobs[torch.arange(len(ys_test_t)), ys_test_t]

avg_nll_trigram_test = nlls.mean().item()


print(f'Test loss = {avg_nll_trigram_test}')


Test loss = 2.4388301372528076


## Summary
Directly indexing into rows of W does not break the pipeline or add unnecessary behind the scene bug in this case. We just need to unsqueeze the output of Weights[idx] to get logits during inference. The training happens normally as it did with one_hot encoding.
The test loss is 2.4388301372528076.